# Control vs. TOU Shifted Residential Load

Spaghetti plot comparing hourly load profiles across **28,195 households** for the control group and TOU rate-shift group.

- **Gray lines / dashes** — control group
- **Orange lines / dashes** — TOU shifted group
- Individual lines: random sample of households (July 1, 2014)
- Bold dashed lines: group average across all 7 days in the dataset
- Shaded band: peak pricing window

**Upload both files when prompted:** `control_profile.csv` and `tou_shifted_profile.zip`

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
SAMPLE_N       = 400           # households shown as spaghetti lines per group
SPAGHETTI_DATE = '2014-07-01'  # date used for individual household lines
PEAK_START     = 17            # peak window start hour (inclusive, 1-24 scale)
PEAK_END       = 19            # peak window end hour (inclusive)
RANDOM_SEED    = 42
OUTPUT_FILE    = 'tou_vs_control_load_profile.png'

In [ ]:
import io
import random
import zipfile

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import files

## 1 · Upload data files

In [ ]:
print('Upload control_profile.csv and tou_shifted_profile.zip')
uploaded = files.upload()

## 2 · Load data

In [ ]:
# Control group
ctrl = pd.read_csv(io.BytesIO(uploaded['control_profile.csv']))

# TOU shifted group — unzip then read whichever CSV is inside
with zipfile.ZipFile(io.BytesIO(uploaded['tou_shifted_profile.zip'])) as z:
    csv_name = next(n for n in z.namelist() if n.endswith('.csv'))
    print(f'Found inside zip: {csv_name}')
    tou = pd.read_csv(z.open(csv_name))

print(f'Control rows : {len(ctrl):,}   |   columns: {list(ctrl.columns)}')
print(f'TOU rows     : {len(tou):,}   |   columns: {list(tou.columns)}')

## 3 · Prepare data

In [ ]:
HOURS = [str(h) for h in range(1, 25)]   # column names '1' … '24'
x     = np.arange(1, 25)                 # numeric x positions

# ── Averages: computed over all available days, all households ───────────────
ctrl_avg = ctrl[HOURS].mean()
tou_avg  = tou[HOURS].mean()

# ── Spaghetti lines: July 1 only, matched sample of households ───────────────
ctrl_day = ctrl[ctrl['date'] == SPAGHETTI_DATE].copy()
tou_day  = tou[tou['date']  == SPAGHETTI_DATE].copy()

shared_hids = list(
    set(ctrl_day['hid'].unique()) & set(tou_day['hid'].unique())
)
print(f'Shared HIDs on {SPAGHETTI_DATE}: {len(shared_hids):,}')

random.seed(RANDOM_SEED)
sample_hids = set(random.sample(shared_hids, min(SAMPLE_N, len(shared_hids))))

ctrl_sample = (
    ctrl_day[ctrl_day['hid'].isin(sample_hids)]
    .set_index('hid')[HOURS]
)
tou_sample = (
    tou_day[tou_day['hid'].isin(sample_hids)]
    .set_index('hid')[HOURS]
)
print(f'Sampled {len(ctrl_sample)} control lines and {len(tou_sample)} TOU lines')

## 4 · Plot

In [ ]:
CTRL_COLOR = '#555555'
TOU_COLOR  = '#E87722'

fig, ax = plt.subplots(figsize=(14, 7))

# ── Individual household lines ────────────────────────────────────────────────
for _, row in ctrl_sample.iterrows():
    ax.plot(x, row.values.astype(float), color='gray', alpha=0.035,
            linewidth=0.5, zorder=1)

for _, row in tou_sample.iterrows():
    ax.plot(x, row.values.astype(float), color=TOU_COLOR, alpha=0.035,
            linewidth=0.5, zorder=1)

# ── Group average lines ───────────────────────────────────────────────────────
ax.plot(x, ctrl_avg.values.astype(float),
        color=CTRL_COLOR, linewidth=2.5, linestyle='--',
        zorder=3, label='Control Average')

ax.plot(x, tou_avg.values.astype(float),
        color=TOU_COLOR, linewidth=2.5, linestyle='--',
        zorder=3, label='TOU Shifted Average')

# ── Peak window shading ───────────────────────────────────────────────────────
ax.axvspan(PEAK_START - 0.5, PEAK_END + 0.5,
           alpha=0.12, color='#FFD700', zorder=0)

# Label sits at top of the shaded band using blended axis transform
# (x in data coords, y in axes [0-1] coords)
blend = ax.get_xaxis_transform()
peak_mid = (PEAK_START + PEAK_END) / 2

def _hour_to_ampm(h):
    """Convert 1-24 hour to '12am', '1am', …, '11pm'."""
    h0 = h - 1
    if h0 == 0:
        return '12am'
    if h0 < 12:
        return f'{h0}am'
    if h0 == 12:
        return '12pm'
    return f'{h0 - 12}pm'

peak_label = (
    f'Peak Window\n'
    f'({_hour_to_ampm(PEAK_START)}–{_hour_to_ampm(PEAK_END + 1)})'
)
ax.text(peak_mid, 0.97, peak_label,
        transform=blend, ha='center', va='top',
        fontsize=9, color='#7a6a00',
        bbox=dict(boxstyle='round,pad=0.3', fc='#fffde0', ec='none', alpha=0.7))

# ── Axes formatting ───────────────────────────────────────────────────────────
tick_hours = range(1, 25, 2)   # every other hour to avoid crowding
ax.set_xticks(list(tick_hours))
ax.set_xticklabels([_hour_to_ampm(h) for h in tick_hours], fontsize=9)
ax.set_xlim(0.5, 24.5)

ax.set_xlabel('Hour of Day', fontsize=11, labelpad=8)
ax.set_ylabel('kWh', fontsize=11, labelpad=8)
ax.set_title('Control vs. TOU Shifted Residential Load',
             fontsize=14, fontweight='bold', pad=12)

ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color='gray',     alpha=0.5, label=f'Control (n={SAMPLE_N} sample)'),
    mpatches.Patch(color=TOU_COLOR,  alpha=0.5, label=f'TOU Shifted (n={SAMPLE_N} sample)'),
    plt.Line2D([0], [0], color=CTRL_COLOR, linewidth=2.5, linestyle='--',
               label='Control Average (all HHs)'),
    plt.Line2D([0], [0], color=TOU_COLOR,  linewidth=2.5, linestyle='--',
               label='TOU Shifted Average (all HHs)'),
    mpatches.Patch(color='#FFD700', alpha=0.4, label='Peak Window'),
]
ax.legend(handles=legend_handles, fontsize=9, framealpha=0.9,
          loc='upper left', borderpad=0.8)

plt.tight_layout()
fig.savefig(OUTPUT_FILE, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUTPUT_FILE}')